# Examples for extended thinking and prompt injection prevention techniques

Example for extended thinking

In [3]:
import json
import anthropic

import os
from dotenv import load_dotenv
load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
model = "claude-sonnet-5-5"

In [22]:
ANALYSIS_SYSTEM = """
You are a contract risk analyst.
Identify the risks in a contract and recommend specific mitigations.
Return a JSON object with:
    - "risk_level": "high"/"medium"/"low"
    - "risks": list of 3 strings (most significant part)
    - "mitigations": list of 3 strings (one per risk, same order)
Return only JSON object.
"""
client =  anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
def analyze_clause(clause: str) -> dict:
    response = client.messages.create(
        model=model,
        max_tokens = 2048,
        thinking = {"type": "adaptive"},
        output_config = {"effort": "high"},
        system = ANALYSIS_SYSTEM,
        messages = [{"role": "user", "content": clause}]
    )

    thinking_words = 0
    result = {}
    print(response)
    for block in response.content:
        print(block.type)
        if block.type == "thinking":
            thinking_words = len(block.thinking.split())
        elif block.type == "text":
            raw = block.text.strip()
            print(raw)
            start = raw.find("{")
            end = raw.find("}") + 1
            print(start, end)
            if start != -1 and start < end:
                result = json.loads(raw[start:end])
    return {"analysis": result, "thinking_words": thinking_words}

In [23]:
print("=== Extended thinking: contract clause risk analysis ===")

clause = (
    "Either party may terminate this agreement immediately upon written notice if the other "
    "party materially breaches any provision and fails to cure such breach within 5 business days "
    "of receiving written notice. Upon termination, all outstanding fees become immediately "
    "due and payable, and the terminated party forfeits any right to refunds or credits."
)

print(f"Clause... {clause[:100]}...\n")

output = analyze_clause(clause=clause)
analysis = output["analysis"]

if analysis:
    print(f"risk analysis: {analysis.get('risk_level')}")
    print(f"risks:")
    for r in analysis.get("risks", []):
        print(f" - {r}")
    print(f"mitigations:")
    for m in analysis.get("mitigations", []):
        print(f" - {m}")

if output['thinking_words'] > 0:
    print(f"\n~{output['thinking_words']} words of internal reasoning visible in the thinking block.")
else:
    print(f"No thinking block created - with adaptive thinking, internal reasoning may not be surfaced even when used.")

=== Extended thinking: contract clause risk analysis ===
Clause... Either party may terminate this agreement immediately upon written notice if the other party materia...

Message(id='msg_011Ch8XF37U95paqC4rMVkiY', container=None, content=[TextBlock(citations=None, text='{\n  "risk_level": "high",\n  "risks": [\n    "Unreasonably short 5-business-day cure period: Complex or systemic breaches (e.g., technical failures, compliance issues, delivery delays) can rarely be remedied in so little time, which makes termination easy to trigger and puts the breaching party at risk of losing the contract over breaches that were curable with reasonable additional time.",\n    "One-sided financial consequences on termination: All outstanding fees become immediately due and the terminated party forfeits all refunds and credits, regardless of breach severity, whether the breach was disputed, or whether prepaid amounts exceed value received. This creates a penalty/forfeiture effect that may be unenforc

In [9]:
print(output)

{'analysis': {}, 'thinking_words': 0}
